# 📓 GOOGLE COLAB NOTEBOOK

## JSON → Corporate PPT Generator

## 🟦 CELL 1 — Install Dependencies

In [ ]:
!pip install python-pptx pydantic

: 

## 🟦 CELL 2 — Imports & Constants

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pathlib import Path
import json

## 🟦 CELL 3 — Theme Definition (LOCKED)

> Single theme used for **all slides**

In [ ]:
THEME = {
    "background": RGBColor(245, 245, 245),
    "title_color": RGBColor(47, 93, 140),
    "text_color": RGBColor(31, 41, 51),
    "muted_text": RGBColor(95, 108, 114),
    "table_header_bg": RGBColor(238, 242, 246),
    "border": RGBColor(217, 221, 225),

    "status": {
        "completed": RGBColor(76, 175, 80),
        "in-progress": RGBColor(74, 144, 226),
        "pending": RGBColor(224, 168, 0)
    },

    "font": {
        "family": "Calibri",
        "title": 32,
        "section": 22,
        "body": 14,
        "small": 11
    }
}

## 🟦 CELL 4 — PPT Helper Functions

In [ ]:
def add_title(slide, text):
    box = slide.shapes.add_textbox(Inches(0.6), Inches(0.4), Inches(9), Inches(1))
    tf = box.text_frame
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(THEME["font"]["title"])
    p.font.bold = True
    p.font.color.rgb = THEME["title_color"]

def add_text(slide, text, top=1.4):
    box = slide.shapes.add_textbox(Inches(0.6), Inches(top), Inches(9), Inches(1))
    tf = box.text_frame
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(THEME["font"]["body"])
    p.font.color.rgb = THEME["text_color"]

def add_banner(slide, text, top=1.2):
    box = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(0.6), Inches(top), Inches(9), Inches(0.7)
    )
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(230, 235, 245)
    box.line.color.rgb = THEME["border"]
    tf = box.text_frame
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(12)

## 🟦 CELL 5 — Table Renderer

In [ ]:
def add_table(slide, table_json, top=2.2):
    rows = len(table_json["rows"]) + 1
    cols = len(table_json["columns"])
    table = slide.shapes.add_table(
        rows, cols,
        Inches(0.6), Inches(top),
        Inches(9), Inches(3)
    ).table

    # Header
    for i, col in enumerate(table_json["columns"]):
        cell = table.cell(0, i)
        cell.text = col["label"]
        cell.text_frame.paragraphs[0].font.bold = True

    # Rows
    for r, row in enumerate(table_json["rows"], start=1):
        for c, col in enumerate(table_json["columns"]):
            val = row[col["key"]]
            table.cell(r, c).text = val

## 🟦 CELL 6 — Card Renderer

In [ ]:
def add_card(slide, title, items, left, top):
    box = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(left), Inches(top), Inches(4.3), Inches(2)
    )
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(255, 255, 255)
    box.line.color.rgb = THEME["border"]

    tf = box.text_frame
    tf.clear()

    p = tf.paragraphs[0]
    p.text = title
    p.font.bold = True

    for item in items:
        p = tf.add_paragraph()
        p.text = f"• {item}"

## 🟦 CELL 7 — Timeline Renderer

In [ ]:
def add_timeline(slide, timeline):
    y = 2
    for phase in timeline["phases"]:
        box = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(1 + phase["start"] * 0.4),
            Inches(y),
            Inches((phase["end"] - phase["start"]) * 0.4),
            Inches(0.5)
        )
        box.text_frame.text = phase["name"]

## 🟦 CELL 8 — FULL JSON INPUT (Random Project Proposal)

> **ONLY this JSON controls the PPT**

In [ ]:
ppt_json = {
  "meta": {
    "title": "Retail Billing Platform Modernization",
    "author": "Ashish",
    "date": "Dec 2025"
  },
  "slides": [

    {
      "type": "cover",
      "title": "Retail Billing Platform Modernization",
      "subtitle": "Project Proposal"
    },

    {
      "type": "content",
      "title": "Project Overview",
      "banner": "This proposal outlines scope, timeline and execution approach.",
      "text": "The goal is to modernize the billing platform for scalability and compliance."
    },

    {
      "type": "table",
      "title": "Project Timeline & Status",
      "table": {
        "columns": [
          {"key": "phase", "label": "Phase"},
          {"key": "timeline", "label": "Timeline"},
          {"key": "status", "label": "Status"}
        ],
        "rows": [
          {"phase": "Design", "timeline": "Week 1–2", "status": "Completed"},
          {"phase": "Development", "timeline": "Week 3–6", "status": "Completed"},
          {"phase": "UAT", "timeline": "Week 7", "status": "In Progress"}
        ]
      }
    },

    {
      "type": "cards",
      "title": "Activities & Risks",
      "cards": [
        {
          "title": "Completed",
          "items": ["UI Design", "Core APIs", "Admin Portal"]
        },
        {
          "title": "Risks",
          "items": ["Stakeholder approval", "UAT feedback"]
        }
      ]
    },

    {
      "type": "timeline",
      "title": "High Level Timeline",
      "timeline": {
        "phases": [
          {"name": "Design", "start": 0, "end": 2},
          {"name": "Build", "start": 2, "end": 6},
          {"name": "UAT", "start": 6, "end": 7}
        ]
      }
    }
  ]
}

## 🟦 CELL 9 — PPT Generator (JSON → PPT)

In [ ]:
prs = Presentation()

for slide_json in ppt_json["slides"]:
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    add_title(slide, slide_json["title"])

    if slide_json["type"] == "content":
        add_banner(slide, slide_json["banner"])
        add_text(slide, slide_json["text"], 2.2)

    if slide_json["type"] == "table":
        add_table(slide, slide_json["table"])

    if slide_json["type"] == "cards":
        add_card(slide, slide_json["cards"][0]["title"],
                 slide_json["cards"][0]["items"], 0.6, 2)
        add_card(slide, slide_json["cards"][1]["title"],
                 slide_json["cards"][1]["items"], 5, 2)

    if slide_json["type"] == "timeline":
        add_timeline(slide, slide_json["timeline"])

## 🟦 CELL 10 — Save & Download PPT

In [ ]:
output_path = "project_proposal.pptx"
prs.save(output_path)
print(f"Presentation saved to: {output_path}")